In [ ]:
# Cell 1: Cài đặt thư viện
!pip install web3 py-solc-x gradio -q
print("✅ Đã cài đặt xong thư viện.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.3/340.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.8/175.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.0 MB/s eta 0:00:00
✅ Đã cài đặt xong thư viện.


In [ ]:
import json, os
from web3 import Web3
from google.colab import files
import gradio as gr

# --- CẤU HÌNH ---
INFURA_URL = "https://sepolia.infura.io/v3/ec3afe35948e4b80a5669c26b6fa3284"
SHOP_ADDRESS = "0x47D821A5B2a86bBECeb07bd05E1Ed32C196E732e"
DEFAULT_ARBITER = "0xc9f30a0287ad97691335281cc27189a95f4b5927"
SELLER_KEY = "aa58650a346dbb6710971f3726aa0618a15d4ca19919d480bd16a5bda38e4675"

w3 = Web3(Web3.HTTPProvider(INFURA_URL))
seller = w3.eth.account.from_key(SELLER_KEY)

if not os.path.exists("ShopABI.json"):
    print("📂 Upload ShopABI.json...")
    files.upload()

with open("ShopABI.json") as f: abi = json.load(f)
contract = w3.eth.contract(address=w3.to_checksum_address(SHOP_ADDRESS), abi=abi)
print(f"✅ Seller sẵn sàng: {seller.address}")

📂 Upload ShopABI.json...


Saving ShopABI.json to ShopABI.json
✅ Seller sẵn sàng: 0xACf3cE5fcEDF60a024A22687683c818f9ef83912


In [ ]:
# --- CELL 2: LOGIC QUẢN LÝ SELLER (FULL TÍNH NĂNG) ---

# 1. Đăng bán (Giữ nguyên)
def create_product(name, price, stock, note):
    try:
        val = w3.to_wei(float(price), "ether")
        print(f"⏳ Đang đăng: {name}...")
        tx = contract.functions.createProduct(
            name, val, int(stock), note, w3.to_checksum_address(DEFAULT_ARBITER)
        ).build_transaction({
            'from': seller.address, 'nonce': w3.eth.get_transaction_count(seller.address),
            'gas': 2000000, 'gasPrice': w3.eth.gas_price
        })
        signed = w3.eth.account.sign_transaction(tx, seller.key)
        w3.eth.wait_for_transaction_receipt(w3.eth.send_raw_transaction(signed.raw_transaction))
        return f"✅ Đã đăng thành công: {name}"
    except Exception as e: return f"❌ Lỗi: {e}"

# 2. Xem kho hàng (Giữ nguyên)
def view_my_inventory():
    res = []
    try:
        c = contract.functions.productCounter().call()
        for i in range(c):
            p = contract.functions.products(i).call()
            if p[5] == seller.address:
                price = w3.from_wei(p[2],'ether')
                res.append(f"🆔 {p[0]} | 📦 {p[1]} | 💰 {price} ETH | 🔢 Kho: {p[3]}")
        return "\n".join(res)
    except: return "Lỗi xem kho"

# 3. Ship hàng (Giữ nguyên)
def ship_order(oid):
    try:
        print(f"⏳ Đang ship đơn {oid}...")
        tx = contract.functions.shipOrder(int(oid)).build_transaction({
            'from': seller.address, 'nonce': w3.eth.get_transaction_count(seller.address),
            'gas': 500000, 'gasPrice': w3.eth.gas_price
        })
        signed = w3.eth.account.sign_transaction(tx, seller.key)
        w3.eth.wait_for_transaction_receipt(w3.eth.send_raw_transaction(signed.raw_transaction))
        return f"✅ Đã Ship đơn {oid}"
    except Exception as e: return f"❌ Lỗi: {e}"

# 4. Xem đơn hàng (Giữ nguyên)
def view_orders():
    res = []
    try:
        c = contract.functions.orderCounter().call()
        for i in range(c):
            o = contract.functions.orders(i).call()
            p = contract.functions.products(o[1]).call()
            if p[5] == seller.address:
                st = ["Chờ Ship", "Đã gửi", "Hoàn tất", "Tranh chấp", "Hoàn tiền"][o[5]]
                res.append(f"Order #{o[0]} | {p[1]} | {st}")
        return "\n".join(res)
    except: return "Lỗi tải đơn"

# 5. (MỚI) LỊCH SỬ BÁN HÀNG
def view_sales_history():
    lines = []
    try:
        count = contract.functions.orderCounter().call()
        header = f"{'MÃ ĐƠN':<8} | {'SẢN PHẨM':<30} | {'KHÁCH HÀNG':<15} | {'TRẠNG THÁI'}"
        lines.append(header)
        lines.append("-" * 80)

        for i in range(count):
            o = contract.functions.orders(i).call() # [0]id, [1]pid, [2]buyer...
            p = contract.functions.products(o[1]).call() # [5]seller

            # Chỉ lấy đơn hàng CỦA MÌNH
            if p[5] == seller.address:
                buyer_short = f"{str(o[2])[:6]}...{str(o[2])[-4:]}"
                st_text = ["Chờ Ship", "Đã Gửi", "✅ Thành Công", "⚖️ Khiếu Nại", "↩️ Đã Hoàn Tiền"][o[5]]
                lines.append(f"#{o[0]:<7} | {p[1]:<30} | {buyer_short:<15} | {st_text}")

        return "\n".join(lines) if len(lines) > 2 else "Chưa có lịch sử bán hàng."
    except Exception as e: return f"❌ Lỗi: {e}"

# 6. (MỚI) THỐNG KÊ DOANH THU
def view_revenue_stats():
    try:
        total_revenue = 0       # Tiền đã về ví (Completed)
        pending_revenue = 0     # Tiền đang treo (Paid, Shipped, Disputed)
        refunded_count = 0      # Số đơn bị hoàn

        count = contract.functions.orderCounter().call()
        for i in range(count):
            o = contract.functions.orders(i).call()
            p = contract.functions.products(o[1]).call()

            if p[5] == seller.address:
                amt = float(w3.from_wei(o[4], 'ether'))
                state = o[5]

                if state == 2: # Completed
                    total_revenue += amt
                elif state == 4: # Refunded
                    refunded_count += 1
                else: # 0, 1, 3 (Treo)
                    pending_revenue += amt

        return (
            f"📊 BÁO CÁO TÀI CHÍNH\n"
            f"----------------------------------\n"
            f"💰 DOANH THU THỰC TẾ (Về ví): {total_revenue:,.4f} ETH\n"
            f"⏳ DOANH THU ĐANG CHỜ       : {pending_revenue:,.4f} ETH\n"
            f"↩️ SỐ ĐƠN BỊ HOÀN TRẢ       : {refunded_count} đơn\n"
            f"----------------------------------\n"
            f"💡 Ghi chú: Chỉ khi đơn hàng ở trạng thái 'Thành Công', tiền mới thực sự thuộc về bạn."
        )
    except Exception as e: return f"❌ Lỗi: {e}"

# 7. (MỚI) XỬ LÝ KHIẾU NẠI (LOGIC 7 NGÀY)
def handle_return_process(order_id, received_goods, days_passed):
    try:
        # Kiểm tra thông tin đơn hàng trên Blockchain
        if not order_id: return "⚠️ Vui lòng nhập Mã đơn hàng."
        o = contract.functions.orders(int(order_id)).call()
        p = contract.functions.products(o[1]).call()

        # Check quyền sở hữu
        if p[5] != seller.address: return "⛔ Đơn hàng này không phải của bạn."

        # Check xem có đang khiếu nại không (State = 3 DISPUTED)
        if o[5] != 3: return f"⚠️ Đơn hàng #{order_id} hiện KHÔNG có khiếu nại nào."

        buyer_reason = o[6] # Lý do khách báo
        days = int(days_passed)

        # --- LOGIC PHÂN XỬ ---
        result_msg = ""

        if received_goods:
            # TRƯỜNG HỢP 1: ĐÃ NHẬN ĐƯỢC HÀNG TRẢ
            if days <= 7:
                result_msg = (
                    f"✅ XÁC NHẬN: Bạn đã nhận lại hàng trong vòng {days} ngày (<= 7 ngày).\n"
                    f"⚖️ PHÁN QUYẾT: Hợp lệ. Trọng tài sẽ xử THẮNG cho NGƯỜI MUA.\n"
                    f"👉 Hành động: Tiền sẽ được hoàn lại cho Buyer."
                )
            else:
                # Thực tế nếu đã nhận được hàng dù quá hạn, thường vẫn hoàn tiền,
                # nhưng theo logic cứng bạn yêu cầu thì cứ nhận hàng là OK.
                # Tuy nhiên, nếu bạn muốn "nhận hàng quá muộn" xử khác thì sửa ở đây.
                # Ở đây tôi để logic: Cứ nhận được hàng là Hoàn tiền cho khách.
                result_msg = (
                    f"⚠️ XÁC NHẬN: Bạn đã nhận lại hàng (dù đã trôi qua {days} ngày).\n"
                    f"⚖️ PHÁN QUYẾT: Đồng thuận trả hàng. Trọng tài sẽ Hoàn tiền cho Buyer."
                )
        else:
            # TRƯỜNG HỢP 2: CHƯA NHẬN ĐƯỢC HÀNG
            if days > 7:
                result_msg = (
                    f"⛔ XÁC NHẬN: Đã quá 7 ngày ({days} ngày) mà Seller chưa nhận được hàng hoàn.\n"
                    f"⚖️ PHÁN QUYẾT: Khách hàng trả hàng thất bại/quá hạn.\n"
                    f"🏆 KẾT QUẢ: Trọng tài sẽ xử THẮNG cho NGƯỜI BÁN (Tiền về ví Seller)."
                )
            else:
                result_msg = (
                    f"⏳ TRẠNG THÁI: Mới trôi qua {days} ngày (Chưa quá 7 ngày).\n"
                    f"✋ HÀNH ĐỘNG: Vui lòng tiếp tục chờ Khách gửi hàng về.\n"
                    f"👉 Chưa thể ra phán quyết lúc này."
                )

        return (
            f"📦 XỬ LÝ KHIẾU NẠI ĐƠN #{order_id}\n"
            f"----------------------------------\n"
            f"Lý do khách báo: '{buyer_reason}'\n"
            f"----------------------------------\n"
            f"{result_msg}"
        )

    except Exception as e: return f"❌ Lỗi xử lý: {e}"

In [ ]:
# --- CELL 3: GIAO DIỆN SELLER (UI UPDATE) ---
import gradio as gr

# Theme Xanh Dương chuyên nghiệp cho Seller
seller_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="indigo",
).set(
    button_primary_background_fill="*primary_600",
)

with gr.Blocks(title="Seller Admin", theme=seller_theme) as app:
    with gr.Row():
        gr.Markdown("# 🏪 KÊNH NGƯỜI BÁN (SELLER CENTER)")

    # --- TAB 1: SẢN PHẨM ---
    with gr.Tab("📦 Quản Lý Kho"):
        with gr.Row():
            with gr.Column(scale=1, variant="panel"):
                gr.Markdown("### ➕ Đăng Sản Phẩm Mới")
                n = gr.Textbox(label="Tên sản phẩm")
                with gr.Row():
                    p = gr.Textbox(label="Giá (ETH)")
                    s = gr.Textbox(label="Số lượng kho")
                nt = gr.Textbox(label="Mô tả / Ghi chú")
                btn_create = gr.Button("Đăng Bán Ngay", variant="primary")
                out_create = gr.Textbox(label="Kết quả đăng")

            with gr.Column(scale=1):
                gr.Markdown("### 📋 Kho Hàng Hiện Tại")
                btn_inv = gr.Button("🔄 Tải Danh Sách Kho")
                out_inv = gr.Textbox(label="Danh sách sản phẩm", lines=12)

        btn_create.click(create_product, [n, p, s, nt], out_create)
        btn_inv.click(view_my_inventory, outputs=out_inv)

    # --- TAB 2: ĐƠN HÀNG ---
    with gr.Tab("🚚 Vận Đơn"):
        gr.Markdown("Theo dõi đơn hàng mới và thực hiện giao hàng.")
        with gr.Row():
            btn_orders = gr.Button("🔄 Cập Nhật Đơn Hàng Mới")
        out_orders = gr.Textbox(label="Danh sách đơn hàng cần xử lý", lines=8)

        with gr.Row(variant="panel"):
            sid = gr.Textbox(label="Nhập Mã Đơn Hàng (Order ID) để giao")
            btn_ship = gr.Button("🚚 XÁC NHẬN GIAO HÀNG (SHIP)", variant="primary")

        out_ship = gr.Textbox(label="Trạng thái giao")

        btn_orders.click(view_orders, outputs=out_orders)
        btn_ship.click(ship_order, sid, out_ship)

    # --- TAB 3: DOANH THU (MỚI) ---
    with gr.Tab("📊 Lịch Sử & Doanh Thu"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("### 📜 Lịch Sử Bán Hàng")
                btn_hist = gr.Button("Xem Lịch Sử Chi Tiết")
                out_hist = gr.Textbox(label="Nhật ký bán hàng", lines=15)
                btn_hist.click(view_sales_history, outputs=out_hist)

            with gr.Column(variant="panel"):
                gr.Markdown("### 💰 Thống Kê Tài Chính")
                gr.Markdown("Tổng hợp doanh thu thực tế và dự kiến.")
                btn_rev = gr.Button("Tính Toán Doanh Thu")
                out_rev = gr.Textbox(label="Báo cáo tài chính", lines=8)
                btn_rev.click(view_revenue_stats, outputs=out_rev)

    # --- TAB 4: KHIẾU NẠI (MỚI - LOGIC 7 NGÀY) ---
    with gr.Tab("🔥 Giải Quyết Khiếu Nại"):
        gr.Markdown("### Quy Trình Xử Lý Hoàn Trả (7 Ngày)")
        gr.Markdown(
            "- Nếu nhận được hàng hoàn trong **7 ngày** -> **Buyer Thắng** (Hoàn tiền).\n"
            "- Nếu quá **7 ngày** không nhận được hàng -> **Seller Thắng** (Giữ tiền)."
        )

        with gr.Row(variant="panel"):
            with gr.Column(scale=1):
                d_oid = gr.Textbox(label="1. Nhập Mã Đơn Khiếu Nại (Order ID)")

                gr.Markdown("---")
                gr.Markdown("**2. Xác nhận từ phía bạn (Seller):**")

                # Input giả lập thời gian để test logic
                d_days = gr.Number(label="⏳ Số ngày đã trôi qua (kể từ khi khách khiếu nại)", value=1)

                # Checkbox xác nhận đã nhận hàng
                d_received = gr.Checkbox(label="✅ TÔI ĐÃ NHẬN ĐƯỢC HÀNG HOÀN TRẢ?", value=False)

                btn_resolve = gr.Button("⚖️ Gửi Xác Nhận & Xem Phán Quyết", variant="stop")

            with gr.Column(scale=1):
                out_dispute = gr.Textbox(label="Kết Quả Xử Lý (Mô phỏng Trọng Tài)", lines=10)

        btn_resolve.click(handle_return_process, [d_oid, d_received, d_days], out_dispute)

app.launch(share=True)

/tmp/ipython-input-2090427774.py:12: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Seller Admin", theme=seller_theme) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://df6c43a1dbed8257a1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# KIỂM TRA SỐ DƯ CỦA 3 TÀI KHOẢN
# ============================================================
from web3 import Web3

# 1. Kết nối
INFURA_URL = "https://sepolia.infura.io/v3/7e77823ee10a44ca81d65804ba33b9c8"
w3 = Web3(Web3.HTTPProvider(INFURA_URL))

if not w3.is_connected():
    print("❌ Không thể kết nối tới Infura")
else:
    print("✅ Đã kết nối mạng Sepolia\n")

# 2. Định nghĩa Private Key (Như đã dùng ở các bước trước)
keys = {
    "NGƯỜI BÁN (Seller) ": "aa58650a346dbb6710971f3726aa0618a15d4ca19919d480bd16a5bda38e4675",
    "NGƯỜI MUA (Buyer)  ": "7f2ebc3b8bb5d73e320c6c38b980c0c1198deee97b7097b74b4b12aa0569ce3e",
    "TRỌNG TÀI (Arbiter)": "68b75cf1c0de408e52091266d0ba8afe0e19d26dc182c3d7c6a1ed5b462a8d5b"
}

print(f"{'VAI TRÒ':<20} | {'ĐỊA CHỈ VÍ':<42} | {'SỐ DƯ (ETH)'}")
print("-" * 80)

# 3. Lặp và lấy số dư
for role, key in keys.items():
    account = w3.eth.account.from_key(key)
    address = account.address

    # Lấy số dư (Wei) và đổi sang ETH
    balance_wei = w3.eth.get_balance(address)

    balance_eth = w3.from_wei(balance_wei, 'ether')


    print(f"{role} | {address} | {balance_eth:.5f} ETH")

print("-" * 80)